In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

data_dir = "/content/drive/MyDrive/전심프/jung/data_split"

batch_size = 16
num_workers = 2  # Windows에서 문제 생기면 0으로 변경

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    root=data_dir + "/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=data_dir + "/val",
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    root=data_dir + "/test",
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

if __name__ == "__main__":
    print("Classes:", train_dataset.classes)
    print("Train:", len(train_dataset))
    print("Val:", len(val_dataset))
    print("Test:", len(test_dataset))

Classes: ['1_level', '3_level', '5_level']
Train: 1490
Val: 318
Test: 323


In [ ]:
# model.py

import torch
import torch.nn as nn
from torchvision import models


def create_model(num_classes=3, pretrained=True):
    """
    EfficientNet-B0 기반 3-class 탈모 단계 분류 모델 생성
    """

    if pretrained:
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    else:
        weights = None

    model = models.efficientnet_b0(weights=weights)

    # EfficientNet-B0의 마지막 classifier를 클래스 수에 맞게 변경
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = create_model(num_classes=3, pretrained=True)
    model = model.to(device)

    print(model)
    print("Model created successfully.")
    print("Device:", device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 221MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [ ]:
#train
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = create_model(num_classes=3, pretrained=True)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

num_epochs = 30
best_val_acc = 0.0

early_stop_patience = 7
early_stop_count = 0

for epoch in range(num_epochs):
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = 100 * train_correct / train_total

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    scheduler.step(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss / len(train_loader):.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Loss: {val_loss / len(val_loader):.4f} "
        f"Val Acc: {val_acc:.2f}% "
        f"LR: {current_lr:.6f}"
    )
    save_path = "/content/drive/MyDrive/전심프/jung/best_model.pth"
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        early_stop_count = 0
        torch.save(model.state_dict(), save_path)
        print("Best model saved!",save_path)
    else:
        early_stop_count += 1
        print(f"Early stopping count: {early_stop_count}/{early_stop_patience}")

    if early_stop_count >= early_stop_patience:
        print("Early stopping triggered.")
        break

print("Training finished.")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Epoch [1/30] Train Loss: 0.7648 Train Acc: 68.19% Val Loss: 0.4840 Val Acc: 80.50% LR: 0.000100
Best model saved! /content/drive/MyDrive/전심프/jung/best_model.pth


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Epoch [2/30] Train Loss: 0.4455 Train Acc: 82.01% Val Loss: 0.4044 Val Acc: 84.91% LR: 0.000100
Best model saved! /content/drive/MyDrive/전심프/jung/best_model.pth


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


...

Epoch [7/30] Train Loss: 0.1682 Train Acc: 94.09% Val Loss: 0.4899 Val Acc: 82.39% LR: 0.000050
Early stopping count: 5/7


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Epoch [8/30] Train Loss: 0.1598 Train Acc: 94.23% Val Loss: 0.4994 Val Acc: 81.45% LR: 0.000050
Early stopping count: 6/7


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Epoch [9/30] Train Loss: 0.1278 Train Acc: 95.50% Val Loss: 0.5359 Val Acc: 80.82% LR: 0.000050
Early stopping count: 7/7
Early stopping triggered.
Training finished.
Best Validation Accuracy: 84.91%


In [ ]:
#지워도됨
import os

save_path = "/content/drive/MyDrive/전심프/jung/best_model.pth"

print("Exists:", os.path.exists(save_path))
print("Files in jung:", os.listdir("/content/drive/MyDrive/전심프/jung"))

Exists: True
Files in jung: ['data_split', 'EfficientNet-B0.ipynb', 'best_model.pth']


In [ ]:
#best model로 val 평가(지워도됨)
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = "/content/drive/MyDrive/전심프/jung/best_model.pth"

check_model = create_model(num_classes=3, pretrained=False)
check_model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
check_model = check_model.to(device)
check_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = check_model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Reloaded Best Model Val Accuracy:", 100 * correct / total)

Reloaded Best Model Val Accuracy: 84.90566037735849


In [ ]:
#성능 test
import torch

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = create_model(num_classes=3, pretrained=False)

# 저장된 모델 경로
model_path = "/content/drive/MyDrive/전심프/jung/best_model.pth"

# 저장된 weight 불러오기
model.load_state_dict(
    torch.load(
        model_path,
        map_location=device,
        weights_only=True
    )
)

model = model.to(device)

# 평가 모드
model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 전체 Accuracy
accuracy = 100 * correct / total

print(f"\nTest Accuracy: {accuracy:.2f}%")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

print("\nConfusion Matrix")
print(cm)

# Classification Report
print("\nClassification Report")

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=test_dataset.classes,
        digits=4
    )
)


Test Accuracy: 85.45%

Confusion Matrix
[[ 67  13   0]
 [ 18 120   2]
 [  0  14  89]]

Classification Report
              precision    recall  f1-score   support

     1_level     0.7882    0.8375    0.8121        80
     3_level     0.8163    0.8571    0.8362       140
     5_level     0.9780    0.8641    0.9175       103

    accuracy                         0.8545       323
   macro avg     0.8609    0.8529    0.8553       323
weighted avg     0.8609    0.8545    0.8562       323



실제 시스템 test

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_names = ["1_level", "3_level", "5_level"]

model_path = "/content/drive/MyDrive/전심프/jung/best_model.pth"

model = create_model(num_classes=3, pretrained=False)
model.load_state_dict(
    torch.load(model_path, map_location=device, weights_only=True)
)
model = model.to(device)
model.eval()


def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")

    input_tensor = test_transform(image)
    input_tensor = input_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probs = F.softmax(outputs, dim=1)[0]

    pred_idx = torch.argmax(probs).item()
    pred_class = class_names[pred_idx]

    print("예측 결과:", pred_class)
    print("\n클래스별 확률")

    for class_name, prob in zip(class_names, probs):
        print(f"{class_name}: {prob.item() * 100:.2f}%")

    return pred_class, probs

In [ ]:
image_path = "/content/drive/MyDrive/전심프/jung/realtestdata/data5.jpg"
predict_image(image_path)

예측 결과: 5_level

클래스별 확률
1_level: 0.89%
3_level: 11.74%
5_level: 87.37%


('5_level', tensor([0.0089, 0.1174, 0.8737], device='cuda:0'))